# Day 92: Chunked Prefill for Long Contexts

**Layer:** Advanced


## Concept Overview

Chunked prefill breaks very long prompts into fixed-size chunks processed iteratively, allowing decode requests to interleave with prefill. This reduces TTFT variance and prevents prefill from blocking decode for hundreds of milliseconds.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Chunked Prefill Scheduler

Simulate chunked prefill interleaved with decode requests and measure latency fairness.


In [ ]:
import numpy as np

def simulate_chunked_prefill(prompt_len=8192, chunk_size=512, decode_reqs=20,
                              prefill_ms_per_tok=0.05, decode_ms_per_tok=20.0):
    """Compare full prefill vs chunked prefill for mixed workload."""
    # Full prefill: all decode blocked until prefill done
    full_prefill_ms = prompt_len * prefill_ms_per_tok
    decode_latencies_full = [full_prefill_ms + i * decode_ms_per_tok
                              for i in range(decode_reqs)]
    
    # Chunked prefill: decode interleaves
    n_chunks = int(np.ceil(prompt_len / chunk_size))
    chunk_ms = chunk_size * prefill_ms_per_tok
    decode_latencies_chunked = []
    for i in range(decode_reqs):
        # Each decode step is delayed by fraction of remaining prefill
        remaining_prefill = max(0, n_chunks - i) * chunk_ms
        decode_latencies_chunked.append(remaining_prefill + decode_ms_per_tok)
    
    return decode_latencies_full, decode_latencies_chunked

full, chunked = simulate_chunked_prefill()
print("Chunked prefill comparison (prompt=8K, 20 decode requests):")
print("{:<25} {:>14} {:>14}".format("Metric", "Full prefill", "Chunked"))
import numpy as np
print("{:<25} {:>13.0f}ms {:>13.0f}ms".format("First decode TTFT", full[0], chunked[0]))
print("{:<25} {:>13.0f}ms {:>13.0f}ms".format("P50 decode latency", np.percentile(full,50), np.percentile(chunked,50)))
print("{:<25} {:>13.0f}ms {:>13.0f}ms".format("P99 decode latency", np.percentile(full,99), np.percentile(chunked,99)))
print()
print("Chunked prefill reduces P99 tail latency for decode requests.")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- Chunked prefill breaks very long prompts into fixed-size chunks processed iteratively, allowing decode requests to interleave with prefill.
- Chunked prefill breaks very long prompts into fixed-size chunks processed iterat....
- Day 92 implementation complete.
